In [ ]:
import numpy as np
import crepe
import soundfile

from audio_preparation import (
    load_audio_file,
    resample_audio,
    normalize_audio,
    segment_audio,
    compute_mel_spectrogram,
    mel_log_compression,
    extract_f0,
)
from drawings import waveform_chart, comparison_chart, spectrogram_chart, f0_chart
from config import Config

In [ ]:
cfg = Config()

# Load Audio

In [ ]:
audio, orig_sr = load_audio_file(
    "YOUR_PATH_TO_AUDIO"
)
print(f"loaded: {len(audio)} samples @ {orig_sr} Hz ({len(audio) / orig_sr:.1f}s)")

In [ ]:
waveform_chart(audio, orig_sr, f"After load — {orig_sr} Hz", color="#6366f1")

# Resample Audio (to 22050 Hz)

In [ ]:
audio_loaded = audio.copy()
audio, new_sr = resample_audio(audio, orig_sr, target_sr=cfg.target_sr)
print(f"resampled: {len(audio)} samples @ {new_sr} Hz")

In [ ]:
comparison_chart(
    [
        (audio_loaded, orig_sr, f"original · {orig_sr} Hz"),
        (audio, new_sr, f"resampled · {new_sr} Hz"),
    ]
)

# Normalise Audio

In [ ]:
audio_pre_norm = audio.copy()
audio = normalize_audio(audio, new_sr, target_lufs=cfg.target_lufs)
print(
    f"normalized: peak {np.abs(audio).max():.3f} (was {np.abs(audio_pre_norm).max():.3f})"
)

In [ ]:
comparison_chart(
    [
        (audio_pre_norm, new_sr, "pre-normalize"),
        (audio, new_sr, "normalized · -23 LUFS"),
    ]
)

# Segment Audio

In [ ]:
segments = segment_audio(audio, new_sr, cfg.window_size)
waveform_chart(segments[0], new_sr, "The first segment", color="#6366f1")

# Spectrograms

In [ ]:
mels = []
for seg in segments:
    mel = compute_mel_spectrogram(
        seg, new_sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length, n_mels=cfg.n_mels
    )
    mel_log_compression(seg)
    mels.append(mel)

## Example Spectrogram

In [ ]:
mel = compute_mel_spectrogram(segments[0], new_sr)
spectrogram_chart(mel, new_sr, title="The spectrogram of the first segment")

In [ ]:
mel = mel_log_compression(mel)
spectrogram_chart(mel, new_sr, title="Log compressed Spectrogram")

# Extract F0 Contours using CREPE

In [ ]:
time, frequency, confidence = extract_f0(
    segments[0], new_sr, model_capacity=cfg.crepe_model_capacity
)
f0_chart(time, frequency, confidence=confidence)

# Save it to example.mp3 in the project's root

In [ ]:
soundfile.write("example.wav", audio, new_sr, format="wav")